# Stage 2 Notebook 76 - Download CLRKDNet GitHub checkpoints

**Purpose.** NB72 self-distilled from NB62 (a student-quality model) and got nothing -- garbage teacher in, garbage student out. This notebook downloads CLRKDNet's published GitHub checkpoint, which scores 80.87 F1 on CULane -- a genuinely strong teacher.

What we pull:
- `dla34_clrnet_culane_8087.pth` (~80 MB): the DLA-34 CLRNet teacher used in the CLRKDNet paper.
- The four training-log .txt files for reference and debugging.

These go into `/content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights/` so they survive Colab session resets. NB77 (joint BDD training with CLRKDNet KD teacher) then consumes `dla34_clrnet_culane_8087.pth`.

Note on architecture mismatch: CLRKDNet's backbone is DLA-34, our model uses RMT-GCA. We do NOT load CLRKDNet's backbone weights into our backbone (incompatible). Instead, we run CLRKDNet as a teacher in inference mode and distill its lane curve outputs into our model via FusionLaneLoss.w_distill. The head architectures differ (CLRNet 192 priors vs our 192-anchor CLRKDLaneHead -- happily very similar) but the OUTPUT format is the same (lane curve coordinates + cls scores), so distillation is well-defined.

### Run mode
1. Run cell 2 first to mount Drive and install dependencies.
2. Run cell 3 to download. ~80 MB over network -- 2-5 minutes typical.
3. Cell 4 verifies the checkpoint loads in PyTorch and prints its key structure.
4. NB77 will reference `/content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights/dla34_clrnet_culane_8087.pth`.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
# Step 1: download CLRKDNet's GitHub release checkpoints.
from pathlib import Path
import os, sys

DEST = '/content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights'
Path(DEST).mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, '-u', 'stage2/scripts/download_clrkdnet_weights.py',
       '--dest', DEST]
LOG_FILE = os.path.join(LOG_DIR, 'clrkdnet_download.log')
print('Downloading CLRKDNet GitHub checkpoints to', DEST)
run_streaming(cmd, log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/download_clrkdnet_weights.py --dest /content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/clrkdnet_download.log
[fetch] CLRKDNet weights into /content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights
[get ] https://github.com/weiqingq/CLRKDNet/releases/download/training_logs/dla34_8087.pth
       -> /content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights/dla34_clrnet_culane_8087.pth
       ... 27.3 MB
       ... 54.5 MB
[done] dla34_clrnet_culane_8087.pth  (63.7 MB)
         CLRNet DLA-34 re-run by authors, 80.87 F1 on CULane (used as the strong teacher in CLRKDNet paper)
[get ] https://github.com/weiqingq/CLRKDNet/releases/download/training_logs/resnet18_distill_log.txt
       -> /content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights/resnet18_distill_log.txt
[done] resnet18_distill_log.txt  (0.1 MB)
         Training log of the distilled 

0

In [3]:
# Step 2: verify the checkpoint structure so NB77 can load it cleanly.
import torch
from pathlib import Path

DEST = '/content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights'
weight_path = Path(DEST) / 'dla34_clrnet_culane_8087.pth'
if not weight_path.exists():
    raise FileNotFoundError(f'Cell 3 did not produce {weight_path}; re-run it.')

sd = torch.load(weight_path, map_location='cpu', weights_only=False)
if isinstance(sd, dict) and 'state_dict' in sd:
    sd = sd['state_dict']
elif isinstance(sd, dict) and 'model' in sd:
    sd = sd['model']
if hasattr(sd, 'state_dict'):
    sd = sd.state_dict()
print(f'top-level type: {type(sd).__name__}')
print(f'num parameters: {len(sd) if hasattr(sd, "__len__") else "<unknown>"}')
print(f'file size: {weight_path.stat().st_size/1e6:.1f} MB')

# Print sample parameter names by prefix family -- helps NB77 build the adapter.
if isinstance(sd, dict):
    families = {}
    for k in sd.keys():
        prefix = k.split('.')[0]
        families.setdefault(prefix, []).append(k)
    print('\nparameter families (prefix -> count, first key):')
    for prefix, keys in sorted(families.items(), key=lambda x: -len(x[1])):
        print(f'  {prefix:25s}  {len(keys):4d} params  e.g. {keys[0]}')

top-level type: dict
num parameters: 5
file size: 63.7 MB

parameter families (prefix -> count, first key):
  net                           1 params  e.g. net
  optim                         1 params  e.g. optim
  scheduler                     1 params  e.g. scheduler
  recorder                      1 params  e.g. recorder
  epoch                         1 params  e.g. epoch


## Notes on next steps

After this notebook runs successfully:
- `/content/drive/MyDrive/EcoCAR/downloads/clrkdnet_weights/dla34_clrnet_culane_8087.pth` exists
- The verification print confirms it's a CLRNet checkpoint (backbone=DLA-34, heads=CLRNet's 192-prior style).

NB77 will:
1. Load the teacher in inference mode (no gradients) on a separate GPU stream.
2. For each training image, run the teacher to get teacher_cls_logits and teacher_coord_pred.
3. Pass them to FusionLaneLoss via the existing `teacher=` argument. `w_distill > 0` triggers the MSE distillation loss between student and teacher outputs.
4. Train the student (our RMT-GCA + CLRKDLaneHead) jointly with BDD det.

Caveat: the teacher was trained on CULane (rural Chinese highway scenes), we're training the student on BDD100K (urban US driving). Distribution shift may limit the teacher's signal, but it should still pull our lane curves toward better geometry.